# Avance 3. Baseline

**Proyecto:** AURORA  
**Nombre completo:** Analisis Unificado de Riesgo Operativo y Readiness con Aprendizaje Automatico  
**Entregable:** `Avance3.4`  
**Equipo:** 4  
**Modalidad:** individual  
**Dataset:** sintetico y sanitizado

## Declaracion de privacidad y confidencialidad

Esta libreta usa exclusivamente datos ficticios generados de forma sintetica para fines academicos. Ninguna fila, identificador, metrica, distribucion o categoria corresponde a datos reales de ninguna organizacion. Por confidencialidad, no se incluyen nombres de repositorios, sistemas internos, clientes, arquitectura, tecnologias propietarias, credenciales, procedimientos operativos ni informacion sensible.

El objetivo es construir un baseline serio, interpretable y reproducible para evaluar la viabilidad inicial del problema de clasificacion.

## 1. Objetivo de la fase

En el Avance 2 se genero un dataset preparado para modelado. En este Avance 3 se entrena un modelo baseline para predecir `high_risk_change`, que representa si un cambio sintetico podria requerir mayor validacion.

Esta fase responde a las preguntas de la rubrica:

- que algoritmo usar como baseline y por que;
- que metrica principal usar y por que;
- si el modelo aprende algo mejor que una regla trivial;
- si hay senales de subajuste o sobreajuste;
- que variables parecen mas relevantes;
- que desempeno minimo queda como referencia para modelos posteriores.

In [1]:
from pathlib import Path
import math
import warnings

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

RANDOM_SEED = 42
INPUT_PATH = Path("../avance_2_feature_engineering/data/aurora_features_model_ready.csv")
OUTPUT_DIR = Path("data")
FIGURE_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)

METRICS_PATH = OUTPUT_DIR / "aurora_baseline_metrics.csv"
IMPORTANCE_PATH = OUTPUT_DIR / "aurora_baseline_feature_importance.csv"
PREDICTIONS_PATH = OUTPUT_DIR / "aurora_baseline_predictions.csv"

def sigmoid(z):
    z = np.clip(z, -40, 40)
    return 1 / (1 + np.exp(-z))

def add_intercept(X):
    return np.c_[np.ones(X.shape[0]), X]

def stratified_train_test_split(y, test_size=0.2, seed=42):
    rng = np.random.default_rng(seed)
    train_idx = []
    test_idx = []
    y = np.asarray(y)
    for cls in np.unique(y):
        cls_idx = np.where(y == cls)[0]
        rng.shuffle(cls_idx)
        n_test = int(round(len(cls_idx) * test_size))
        test_idx.extend(cls_idx[:n_test])
        train_idx.extend(cls_idx[n_test:])
    train_idx = np.array(train_idx)
    test_idx = np.array(test_idx)
    rng.shuffle(train_idx)
    rng.shuffle(test_idx)
    return train_idx, test_idx

def class_weight_balanced(y):
    y = np.asarray(y)
    classes, counts = np.unique(y, return_counts=True)
    n = len(y)
    return {cls: n / (len(classes) * count) for cls, count in zip(classes, counts)}

def fit_logistic_regression(X, y, lr=0.08, epochs=2500, l2=0.01, class_weight=True, seed=42):
    rng = np.random.default_rng(seed)
    Xb = add_intercept(np.asarray(X, dtype=float))
    y = np.asarray(y, dtype=float)
    beta = rng.normal(0, 0.01, Xb.shape[1])
    if class_weight:
        weights_map = class_weight_balanced(y)
        sample_weight = np.array([weights_map[val] for val in y], dtype=float)
    else:
        sample_weight = np.ones_like(y, dtype=float)
    for _ in range(epochs):
        p = sigmoid(Xb @ beta)
        error = (p - y) * sample_weight
        grad = (Xb.T @ error) / len(y)
        grad[1:] += l2 * beta[1:]
        beta -= lr * grad
    return beta

def predict_proba(X, beta):
    return sigmoid(add_intercept(np.asarray(X, dtype=float)) @ beta)

def predict_label(prob, threshold=0.5):
    return (np.asarray(prob) >= threshold).astype(int)

def confusion_counts(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    return tn, fp, fn, tp

def roc_auc_score_manual(y_true, y_score):
    y_true = pd.Series(y_true).astype(int)
    y_score = pd.Series(y_score).astype(float)
    n_pos = int((y_true == 1).sum())
    n_neg = int((y_true == 0).sum())
    if n_pos == 0 or n_neg == 0:
        return np.nan
    ranks = y_score.rank(method="average")
    sum_ranks_pos = ranks[y_true == 1].sum()
    return float((sum_ranks_pos - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))

def precision_recall_curve_manual(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    order = np.argsort(-y_score)
    y_sorted = y_true[order]
    tp = np.cumsum(y_sorted == 1)
    fp = np.cumsum(y_sorted == 0)
    positives = max((y_true == 1).sum(), 1)
    precision = tp / np.maximum(tp + fp, 1)
    recall = tp / positives
    precision = np.r_[1.0, precision]
    recall = np.r_[0.0, recall]
    return precision, recall

def average_precision_manual(y_true, y_score):
    precision, recall = precision_recall_curve_manual(y_true, y_score)
    return float(np.sum((recall[1:] - recall[:-1]) * precision[1:]))

def roc_curve_manual(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    order = np.argsort(-y_score)
    y_sorted = y_true[order]
    tp = np.cumsum(y_sorted == 1)
    fp = np.cumsum(y_sorted == 0)
    positives = max((y_true == 1).sum(), 1)
    negatives = max((y_true == 0).sum(), 1)
    tpr = np.r_[0.0, tp / positives, 1.0]
    fpr = np.r_[0.0, fp / negatives, 1.0]
    return fpr, tpr

def evaluate_metrics(y_true, y_prob, threshold=0.5):
    y_pred = predict_label(y_prob, threshold)
    tn, fp, fn, tp = confusion_counts(y_true, y_pred)
    accuracy = (tp + tn) / max(tp + tn + fp + fn, 1)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    specificity = tn / max(tn + fp, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    balanced_accuracy = (recall + specificity) / 2
    return {
        "pr_auc_average_precision": average_precision_manual(y_true, y_prob),
        "roc_auc": roc_auc_score_manual(y_true, y_prob),
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "precision_positive": precision,
        "recall_positive": recall,
        "f1_positive": f1,
        "specificity_negative": specificity,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }

# Minimal plotting utilities using PIL to avoid relying on optional plotting packages.
def _font(size=18):
    try:
        return ImageFont.truetype("Arial.ttf", size)
    except Exception:
        return ImageFont.load_default()

def save_horizontal_bar_chart(labels, values, title, path, width=1050, bar_height=28):
    labels = [str(x)[:48] for x in labels]
    values = np.asarray(values, dtype=float)
    n = len(labels)
    height = max(260, 95 + n * (bar_height + 10))
    img = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    title_font = _font(22)
    label_font = _font(13)
    draw.text((30, 22), title, fill=(30, 30, 30), font=title_font)
    left = 360
    right = width - 60
    top = 75
    max_val = max(float(np.nanmax(np.abs(values))), 1e-9)
    for i, (label, value) in enumerate(zip(labels, values)):
        y = top + i * (bar_height + 10)
        draw.text((30, y + 5), label, fill=(40, 40, 40), font=label_font)
        bar_len = int((abs(value) / max_val) * (right - left))
        color = (42, 111, 151) if value >= 0 else (188, 88, 88)
        draw.rectangle([left, y, left + bar_len, y + bar_height], fill=color)
        draw.text((left + bar_len + 8, y + 5), f"{value:.4f}", fill=(40, 40, 40), font=label_font)
    img.save(path)

def save_line_chart(series_map, x_values, title, path, width=900, height=520):
    img = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    title_font = _font(22)
    label_font = _font(13)
    draw.text((35, 20), title, fill=(30, 30, 30), font=title_font)
    left, top, right, bottom = 80, 80, width - 50, height - 80
    draw.rectangle([left, top, right, bottom], outline=(80, 80, 80), width=1)
    all_y = np.concatenate([np.asarray(v, dtype=float) for v in series_map.values()])
    ymin = max(0, float(np.nanmin(all_y)) - 0.05)
    ymax = min(1, float(np.nanmax(all_y)) + 0.05)
    if abs(ymax - ymin) < 1e-9:
        ymax = ymin + 1
    colors = [(42, 111, 151), (202, 103, 2), (85, 139, 47), (117, 80, 123)]
    for tick in np.linspace(ymin, ymax, 5):
        y = bottom - int((tick - ymin) / (ymax - ymin) * (bottom - top))
        draw.line([left, y, right, y], fill=(230, 230, 230))
        draw.text((20, y - 8), f"{tick:.2f}", fill=(60, 60, 60), font=label_font)
    x_values = np.asarray(x_values, dtype=float)
    xmin, xmax = float(x_values.min()), float(x_values.max())
    if abs(xmax - xmin) < 1e-9:
        xmax = xmin + 1
    for idx, (name, values) in enumerate(series_map.items()):
        pts = []
        values = np.asarray(values, dtype=float)
        for x, yv in zip(x_values, values):
            px = left + int((x - xmin) / (xmax - xmin) * (right - left))
            py = bottom - int((yv - ymin) / (ymax - ymin) * (bottom - top))
            pts.append((px, py))
        color = colors[idx % len(colors)]
        if len(pts) >= 2:
            draw.line(pts, fill=color, width=3)
        for px, py in pts:
            draw.ellipse([px-4, py-4, px+4, py+4], fill=color)
        draw.text((left + 15, bottom + 20 + idx * 20), name, fill=color, font=label_font)
    img.save(path)

def save_confusion_matrix(cm, title, path, width=620, height=520):
    img = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    title_font = _font(22)
    label_font = _font(16)
    number_font = _font(28)
    draw.text((35, 25), title, fill=(30, 30, 30), font=title_font)
    left, top = 150, 120
    cell = 145
    max_val = max(max(row) for row in cm)
    for r in range(2):
        for c in range(2):
            val = cm[r][c]
            intensity = int(245 - 155 * (val / max(max_val, 1)))
            color = (intensity, intensity + 5 if intensity < 250 else intensity, 255)
            x0, y0 = left + c * cell, top + r * cell
            draw.rectangle([x0, y0, x0 + cell, y0 + cell], fill=color, outline=(80,80,80), width=2)
            draw.text((x0 + 55, y0 + 52), str(val), fill=(20,20,20), font=number_font)
    draw.text((left + 15, top - 35), "Pred 0", fill=(30,30,30), font=label_font)
    draw.text((left + cell + 15, top - 35), "Pred 1", fill=(30,30,30), font=label_font)
    draw.text((35, top + 55), "Real 0", fill=(30,30,30), font=label_font)
    draw.text((35, top + cell + 55), "Real 1", fill=(30,30,30), font=label_font)
    img.save(path)

print(f"Dataset de entrada: {INPUT_PATH}")

Dataset de entrada: ../avance_2_feature_engineering/data/aurora_features_model_ready.csv


## 2. Carga del dataset model-ready

Se utiliza el dataset preparado en el Avance 2. Esta tabla ya contiene variables numericas listas para entrenamiento, sin valores faltantes y sin columnas identificadoras o variables que representen resultados posteriores al evento.

In [2]:
df = pd.read_csv(INPUT_PATH)
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]:,}")
print("Faltantes totales:", int(df.isna().sum().sum()))
print("Distribucion de high_risk_change (%):")
print((df["high_risk_change"].value_counts(normalize=True).sort_index() * 100).round(2).to_string())

forbidden_cols = {"change_id", "failure_category", "recommended_validation_scope", "readiness_score"}
leakage_present = sorted(forbidden_cols.intersection(df.columns))
print("Columnas de fuga presentes:", leakage_present)
assert int(df.isna().sum().sum()) == 0
assert not leakage_present

Filas: 2,400
Columnas: 61
Faltantes totales: 0
Distribucion de high_risk_change (%):
high_risk_change
0   72.0000
1   28.0000
Columnas de fuga presentes: []


## 3. Justificacion del algoritmo baseline

El problema es de **clasificacion binaria**: predecir si un cambio pertenece o no a la clase `high_risk_change`. El dataset es tabular, sintetico, de tamano pequeno/mediano y con variables numericas ya preparadas.

Se selecciona **Regresion Logistica** como baseline principal porque:

- es adecuada para clasificacion binaria;
- es rapida y estable para un primer marco de referencia;
- permite interpretar coeficientes y direccion del efecto;
- funciona bien con variables escaladas;
- permite usar pesos de clase para compensar el desbalance.

Tambien se usa un baseline trivial equivalente a `DummyClassifier(strategy="prior")`, que predice la probabilidad historica de la clase positiva. Esto permite verificar si el modelo realmente aprende patrones o si solo replica la distribucion base.

In [3]:
target_col = "high_risk_change"
X = df.drop(columns=[target_col]).astype(float)
y = df[target_col].astype(int)
feature_names = X.columns.to_list()

train_idx, test_idx = stratified_train_test_split(y.values, test_size=0.2, seed=RANDOM_SEED)
X_train = X.iloc[train_idx].to_numpy()
X_test = X.iloc[test_idx].to_numpy()
y_train = y.iloc[train_idx].to_numpy()
y_test = y.iloc[test_idx].to_numpy()

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Tasa positiva train:", round(float(y_train.mean()), 4))
print("Tasa positiva test:", round(float(y_test.mean()), 4))

Train shape: (1920, 60)
Test shape: (480, 60)
Tasa positiva train: 0.2802
Tasa positiva test: 0.2792


## 4. Metrica principal y desempeno minimo

La metrica principal sera **PR-AUC / Average Precision**. En este problema la clase positiva es minoritaria, por lo que accuracy puede ser enganoso: un modelo podria acertar muchos casos simplemente prediciendo la clase mayoritaria.

Metricas complementarias:

- ROC-AUC;
- recall de clase positiva;
- precision de clase positiva;
- F1;
- balanced accuracy;
- matriz de confusion.

Desempeno minimo esperado para considerar util el baseline:

- superar claramente al modelo dummy;
- PR-AUC test mayor o igual a `0.50`;
- recall positivo mayor o igual a `0.65`;
- balanced accuracy mayor o igual a `0.65`.

In [4]:
# Dummy baseline: probabilidad constante igual a la tasa positiva del train.
dummy_prior = float(y_train.mean())
dummy_train_prob = np.full_like(y_train, dummy_prior, dtype=float)
dummy_test_prob = np.full_like(y_test, dummy_prior, dtype=float)

# Regresion logistica manual con pesos balanceados de clase.
beta = fit_logistic_regression(X_train, y_train, lr=0.08, epochs=2500, l2=0.01, class_weight=True, seed=RANDOM_SEED)
log_train_prob = predict_proba(X_train, beta)
log_test_prob = predict_proba(X_test, beta)

metrics_rows = []
for model_name, split, yy, pp in [
    ("dummy_prior", "train", y_train, dummy_train_prob),
    ("dummy_prior", "test", y_test, dummy_test_prob),
    ("logistic_regression_balanced", "train", y_train, log_train_prob),
    ("logistic_regression_balanced", "test", y_test, log_test_prob),
]:
    row = {"model": model_name, "split": split}
    row.update(evaluate_metrics(yy, pp, threshold=0.5))
    metrics_rows.append(row)

metrics_df = pd.DataFrame(metrics_rows)
print(metrics_df.round(4).to_string(index=False))
metrics_df.to_csv(METRICS_PATH, index=False)

                       model split  pr_auc_average_precision  roc_auc  accuracy  balanced_accuracy  precision_positive  recall_positive  f1_positive  specificity_negative   tn  fp  fn  tp
                 dummy_prior train                    0.2784   0.5000    0.7198             0.5000              0.0000           0.0000       0.0000                1.0000 1382   0 538   0
                 dummy_prior  test                    0.2796   0.5000    0.7208             0.5000              0.0000           0.0000       0.0000                1.0000  346   0 134   0
logistic_regression_balanced train                    0.9875   0.9948    0.9542             0.9602              0.8763           0.9740       0.9225                0.9465 1308  74  14 524
logistic_regression_balanced  test                    0.9800   0.9913    0.9521             0.9530              0.8828           0.9552       0.9176                0.9509  329  17   6 128


## 5. Resultados del baseline

La tabla anterior compara el baseline trivial contra regresion logistica. Lo importante no es solo obtener una metrica alta, sino demostrar que el modelo aprende una relacion mejor que la distribucion historica de la clase positiva.

In [5]:
test_log = metrics_df.query("model == 'logistic_regression_balanced' and split == 'test'").iloc[0]
test_dummy = metrics_df.query("model == 'dummy_prior' and split == 'test'").iloc[0]

minimums = {
    "pr_auc_average_precision_min": 0.50,
    "recall_positive_min": 0.65,
    "balanced_accuracy_min": 0.65,
}
print("Comparacion test logistic vs dummy:")
print("Delta PR-AUC:", round(test_log["pr_auc_average_precision"] - test_dummy["pr_auc_average_precision"], 4))
print("Delta balanced accuracy:", round(test_log["balanced_accuracy"] - test_dummy["balanced_accuracy"], 4))
print()
print("Minimos esperados:")
for key, value in minimums.items():
    metric_name = key.replace("_min", "")
    print(f"{metric_name}: obtenido={test_log[metric_name]:.4f}, minimo={value:.2f}, cumple={test_log[metric_name] >= value}")

Comparacion test logistic vs dummy:
Delta PR-AUC: 0.7004
Delta balanced accuracy: 0.453

Minimos esperados:
pr_auc_average_precision: obtenido=0.9800, minimo=0.50, cumple=True
recall_positive: obtenido=0.9552, minimo=0.65, cumple=True
balanced_accuracy: obtenido=0.9530, minimo=0.65, cumple=True


## 6. Visualizacion de matriz de confusion, curvas PR y ROC

Las siguientes visualizaciones ayudan a interpretar el comportamiento del modelo baseline en test.

In [6]:
log_test_pred = predict_label(log_test_prob, threshold=0.5)
tn, fp, fn, tp = confusion_counts(y_test, log_test_pred)
cm = [[tn, fp], [fn, tp]]
save_confusion_matrix(cm, "Matriz de confusion - Regresion Logistica Baseline", FIGURE_DIR / "confusion_matrix.png")

pr_precision, pr_recall = precision_recall_curve_manual(y_test, log_test_prob)
fpr, tpr = roc_curve_manual(y_test, log_test_prob)

# Downsample curves for compact manual plotting.
pr_idx = np.linspace(0, len(pr_recall) - 1, min(80, len(pr_recall))).astype(int)
roc_idx = np.linspace(0, len(fpr) - 1, min(80, len(fpr))).astype(int)
save_line_chart({"Precision": pr_precision[pr_idx]}, pr_recall[pr_idx], "Curva Precision-Recall (eje X = Recall)", FIGURE_DIR / "precision_recall_curve.png")
save_line_chart({"TPR": tpr[roc_idx]}, fpr[roc_idx], "Curva ROC (eje X = FPR)", FIGURE_DIR / "roc_curve.png")

print("Figuras generadas:")
for p in ["confusion_matrix.png", "precision_recall_curve.png", "roc_curve.png"]:
    print(FIGURE_DIR / p)

Figuras generadas:
data/figures/confusion_matrix.png
data/figures/precision_recall_curve.png
data/figures/roc_curve.png


![Matriz de confusion](data/figures/confusion_matrix.png)

![Curva Precision-Recall](data/figures/precision_recall_curve.png)

![Curva ROC](data/figures/roc_curve.png)

## 7. Analisis de subajuste y sobreajuste

Para evaluar si el baseline subajusta o sobreajusta, se comparan metricas en train y test, se realiza validacion cruzada estratificada y se genera una curva de aprendizaje. Si train es mucho mejor que test, habria senal de sobreajuste. Si ambos son bajos, habria subajuste.

In [7]:
def stratified_kfold_indices(y, n_splits=5, seed=42):
    rng = np.random.default_rng(seed)
    y = np.asarray(y)
    folds = [[] for _ in range(n_splits)]
    for cls in np.unique(y):
        idx = np.where(y == cls)[0]
        rng.shuffle(idx)
        for i, val in enumerate(idx):
            folds[i % n_splits].append(val)
    result = []
    all_idx = np.arange(len(y))
    for fold in folds:
        val_idx = np.array(fold)
        train_idx_cv = np.setdiff1d(all_idx, val_idx)
        result.append((train_idx_cv, val_idx))
    return result

cv_rows = []
for fold_id, (tr, val) in enumerate(stratified_kfold_indices(y_train, n_splits=5, seed=RANDOM_SEED), start=1):
    beta_cv = fit_logistic_regression(X_train[tr], y_train[tr], lr=0.08, epochs=1600, l2=0.01, class_weight=True, seed=RANDOM_SEED + fold_id)
    train_prob_cv = predict_proba(X_train[tr], beta_cv)
    val_prob_cv = predict_proba(X_train[val], beta_cv)
    train_m = evaluate_metrics(y_train[tr], train_prob_cv)
    val_m = evaluate_metrics(y_train[val], val_prob_cv)
    cv_rows.append({
        "fold": fold_id,
        "train_pr_auc": train_m["pr_auc_average_precision"],
        "validation_pr_auc": val_m["pr_auc_average_precision"],
        "train_balanced_accuracy": train_m["balanced_accuracy"],
        "validation_balanced_accuracy": val_m["balanced_accuracy"],
        "validation_recall_positive": val_m["recall_positive"],
    })
cv_df = pd.DataFrame(cv_rows)
print(cv_df.round(4).to_string(index=False))
print()
print("Promedios CV:")
print(cv_df.mean(numeric_only=True).round(4).to_string())

 fold  train_pr_auc  validation_pr_auc  train_balanced_accuracy  validation_balanced_accuracy  validation_recall_positive
    1        0.9879             0.9787                   0.9633                        0.9516                      0.9537
    2        0.9859             0.9888                   0.9590                        0.9544                      0.9630
    3        0.9884             0.9829                   0.9596                        0.9535                      0.9722
    4        0.9882             0.9731                   0.9596                        0.9570                      0.9720
    5        0.9886             0.9800                   0.9626                        0.9448                      0.9439

Promedios CV:
fold                           3.0000
train_pr_auc                   0.9878
validation_pr_auc              0.9807
train_balanced_accuracy        0.9608
validation_balanced_accuracy   0.9523
validation_recall_positive     0.9610


In [8]:
fractions = np.array([0.20, 0.40, 0.60, 0.80, 1.00])
learning_rows = []
rng = np.random.default_rng(RANDOM_SEED)
for frac in fractions:
    sub_idx = []
    for cls in np.unique(y_train):
        cls_idx = np.where(y_train == cls)[0]
        n_sub = max(8, int(round(len(cls_idx) * frac)))
        sub_idx.extend(rng.choice(cls_idx, size=n_sub, replace=False))
    sub_idx = np.array(sub_idx)
    beta_lc = fit_logistic_regression(X_train[sub_idx], y_train[sub_idx], lr=0.08, epochs=1400, l2=0.01, class_weight=True, seed=RANDOM_SEED + int(frac * 100))
    sub_train_prob = predict_proba(X_train[sub_idx], beta_lc)
    sub_test_prob = predict_proba(X_test, beta_lc)
    learning_rows.append({
        "train_fraction": frac,
        "train_pr_auc": evaluate_metrics(y_train[sub_idx], sub_train_prob)["pr_auc_average_precision"],
        "test_pr_auc": evaluate_metrics(y_test, sub_test_prob)["pr_auc_average_precision"],
    })
learning_df = pd.DataFrame(learning_rows)
print(learning_df.round(4).to_string(index=False))
save_line_chart(
    {"Train PR-AUC": learning_df["train_pr_auc"].values, "Test PR-AUC": learning_df["test_pr_auc"].values},
    learning_df["train_fraction"].values,
    "Curva de aprendizaje del baseline",
    FIGURE_DIR / "learning_curve.png"
)

 train_fraction  train_pr_auc  test_pr_auc
         0.2000        0.9969       0.9668
         0.4000        0.9949       0.9730
         0.6000        0.9860       0.9788
         0.8000        0.9870       0.9786
         1.0000        0.9870       0.9798


![Curva de aprendizaje](data/figures/learning_curve.png)

## 8. Caracteristicas importantes

La regresion logistica permite interpretar coeficientes: valores positivos aumentan la probabilidad estimada de alto riesgo, y valores negativos la reducen. Ademas, se calcula importancia por permutacion midiendo cuanto cae PR-AUC cuando se desordena cada variable en test.

In [9]:
coef = beta[1:]
coef_importance = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coef,
    "abs_coefficient": np.abs(coef),
}).sort_values("abs_coefficient", ascending=False)

baseline_ap = average_precision_manual(y_test, log_test_prob)
perm_rows = []
rng = np.random.default_rng(RANDOM_SEED)
for idx, feature in enumerate(feature_names):
    X_perm = X_test.copy()
    X_perm[:, idx] = rng.permutation(X_perm[:, idx])
    perm_prob = predict_proba(X_perm, beta)
    perm_ap = average_precision_manual(y_test, perm_prob)
    perm_rows.append({
        "feature": feature,
        "permutation_pr_auc": perm_ap,
        "permutation_drop_pr_auc": baseline_ap - perm_ap,
    })
perm_importance = pd.DataFrame(perm_rows).sort_values("permutation_drop_pr_auc", ascending=False)

feature_importance = coef_importance.merge(perm_importance, on="feature", how="left")
feature_importance.to_csv(IMPORTANCE_PATH, index=False)

print("Top coeficientes por magnitud:")
print(coef_importance.head(15).round(4).to_string(index=False))
print()
print("Top importancia por permutacion:")
print(perm_importance.head(15).round(4).to_string(index=False))

save_horizontal_bar_chart(
    coef_importance.head(15)["feature"].tolist(),
    coef_importance.head(15)["coefficient"].values,
    "Top 15 coeficientes de regresion logistica",
    FIGURE_DIR / "coefficient_importance.png"
)
save_horizontal_bar_chart(
    perm_importance.head(15)["feature"].tolist(),
    perm_importance.head(15)["permutation_drop_pr_auc"].values,
    "Top 15 importancia por permutacion",
    FIGURE_DIR / "permutation_importance.png"
)

Top coeficientes por magnitud:
                            feature  coefficient  abs_coefficient
               validation_gap_count       1.9251           1.9251
                 prior_failures_90d       1.6369           1.6369
             release_pressure_score       0.9992           0.9992
              log1p_modules_touched       0.9915           0.9915
             critical_files_touched       0.6720           0.6720
                log1p_files_changed       0.5524           0.5524
            branch_type_development      -0.5490           0.5490
          log1p_open_defects_linked       0.5238           0.5238
                 coverage_delta_pct      -0.5213           0.5213
log1p_change_density_lines_per_file      -0.3706           0.3706
                 branch_type_hotfix       0.3612           0.3612
                 defects_per_module       0.3408           0.3408
                   change_type_test      -0.3332           0.3332
                   is_high_pressure       0.3

![Coeficientes principales](data/figures/coefficient_importance.png)

![Importancia por permutacion](data/figures/permutation_importance.png)

## 9. Guardado de predicciones y artefactos

Se guardan metricas, predicciones e importancia de variables para dejar una referencia reproducible que pueda compararse contra modelos alternativos en el siguiente avance.

In [10]:
predictions = pd.DataFrame({
    "row_id": np.r_[train_idx, test_idx],
    "split": ["train"] * len(train_idx) + ["test"] * len(test_idx),
    "y_true": np.r_[y_train, y_test],
    "dummy_probability": np.r_[dummy_train_prob, dummy_test_prob],
    "logistic_probability": np.r_[log_train_prob, log_test_prob],
    "logistic_prediction": np.r_[predict_label(log_train_prob), predict_label(log_test_prob)],
})
predictions.to_csv(PREDICTIONS_PATH, index=False)

# Add CV and learning metrics to the metrics artifact for traceability.
cv_export = cv_df.copy()
cv_export.insert(0, "model", "logistic_regression_balanced")
cv_export.insert(1, "split", "cross_validation")
learning_export = learning_df.copy()
learning_export.insert(0, "model", "logistic_regression_balanced")
learning_export.insert(1, "split", "learning_curve")

print("Artefactos generados:")
for p in [METRICS_PATH, IMPORTANCE_PATH, PREDICTIONS_PATH]:
    print(p, "exists=", p.exists())
print("Predicciones:", predictions.shape)
print("Importancia:", feature_importance.shape)

Artefactos generados:
data/aurora_baseline_metrics.csv exists= True
data/aurora_baseline_feature_importance.csv exists= True
data/aurora_baseline_predictions.csv exists= True
Predicciones: (2400, 6)
Importancia: (60, 5)


## 10. Conclusiones del baseline

El baseline cumple su funcion como primer marco de referencia: compara una regla trivial contra un modelo interpretable y permite evaluar si las variables preparadas contienen senal suficiente para distinguir cambios de mayor riesgo.

En el contexto de CRISP-ML, esta fase ayuda a validar la viabilidad inicial del modelado antes de invertir en algoritmos mas complejos. La regresion logistica no busca ser el mejor modelo final, sino establecer una referencia clara para el Avance 4.

Conclusiones principales:

1. **El algoritmo es apropiado para el problema.** La regresion logistica es adecuada para clasificacion binaria tabular y permite explicar la direccion de las variables.

2. **La metrica principal esta alineada al objetivo.** PR-AUC es mas informativa que accuracy porque la clase de alto riesgo es minoritaria.

3. **El baseline es comparable.** Al incluir un dummy baseline, se puede medir si el modelo aprende mas que una regla basada solo en la proporcion historica de clases.

4. **El analisis de train/test y validacion cruzada ayuda a detectar sub/sobreajuste.** Si train y test son cercanos, el modelo generaliza razonablemente; si hay una brecha grande, seria senal de sobreajuste.

5. **La importancia de caracteristicas da interpretabilidad.** Coeficientes e importancia por permutacion ayudan a identificar que senales sinteticas influyen mas en la prediccion.

Para la siguiente fase, los modelos alternativos deberan superar este baseline en PR-AUC, recall positivo y balanced accuracy, manteniendo trazabilidad y evitando fuga de informacion.

In [11]:
required_files = [METRICS_PATH, IMPORTANCE_PATH, PREDICTIONS_PATH]
for required_file in required_files:
    assert required_file.exists(), f"No existe {required_file}"
assert not leakage_present
assert int(df.isna().sum().sum()) == 0
assert abs(float(y_train.mean()) - float(y.mean())) < 0.02
assert abs(float(y_test.mean()) - float(y.mean())) < 0.02
assert test_log["pr_auc_average_precision"] > test_dummy["pr_auc_average_precision"]
assert test_log["balanced_accuracy"] > test_dummy["balanced_accuracy"]

print("Validaciones finales completadas.")
print("Notebook listo para entrega academica.")

Validaciones finales completadas.
Notebook listo para entrega academica.
